## Setup — run this first

Mounts Drive, points the notebook at your project folder, and installs
what's missing. No git, no tokens.

**Your Drive folder must look like this:**

```
MyDrive/Ghana_Dropout_Project_R02/
├── config.py          <- these three at the TOP level,
├── losses.py             not inside notebooks/
├── pipeline.py
├── requirements.txt
├── notebooks/         <- the 11 notebooks
└── data-raw/
    └── ghana_dropout_study_M.xlsx
```

`results/`, `figures/`, `models/` and `data-processed/` are created for you.

Drive saves as it goes, so there is nothing to push — but see the checklist
in the last cell before you submit.


In [1]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 17.6 MB/s eta 0:00:00


In [2]:
# ============================================================
# SETUP — Google Drive. Run first. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

PROJECT = "/content/drive/MyDrive/Ghana_Dropout_Project_R02"   # <-- edit if yours differs
RAW_XLSX_NAME = "ghana_dropout_study_M.xlsx"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    root = Path(PROJECT)
    if not root.exists():
        raise FileNotFoundError(
            f"{PROJECT} does not exist.\n"
            "Create that folder in My Drive and put config.py, losses.py, "
            "pipeline.py, requirements.txt, the notebooks/ folder and "
            "data-raw/ inside it."
        )

    # the three modules must sit at the project root, not in notebooks/
    missing = [m for m in ("config.py", "losses.py", "pipeline.py")
               if not (root / m).exists()]
    if missing:
        stray = [m for m in missing if (root / "notebooks" / m).exists()]
        msg = f"Missing from {PROJECT}: {missing}"
        if stray:
            msg += (f"\n{stray} are in notebooks/ instead. Move them UP one "
                    "level, into the project folder itself. If they stay in "
                    "notebooks/, that folder gets treated as the project root "
                    "and results/ is written in the wrong place.")
        raise FileNotFoundError(msg)

    os.chdir(root)
    os.environ["DROPOUT_REPO"] = str(root)

    # Forget any previously loaded copy of the project modules. Python keeps
    # the first version it imported for the whole session, so an edited
    # config.py is silently ignored until the runtime restarts. This makes
    # every run use the files currently in Drive.
    for _m in ("config", "losses", "pipeline"):
        sys.modules.pop(_m, None)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    # ---- dependencies: only install what is actually missing ------------
    need = []
    for mod, pkg in [("lightgbm", "lightgbm"), ("shap", "shap"),
                     ("catboost", "catboost"), ("xgboost", "xgboost"),
                     ("imblearn", "imbalanced-learn"), ("openpyxl", "openpyxl")]:
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print("installing:", need)
        subprocess.run(f"pip install -q {' '.join(need)}", shell=True)
    else:
        print("all dependencies present")

    # ---- raw workbook ---------------------------------------------------
    (root / "data-raw").mkdir(exist_ok=True)
    xlsx = root / "data-raw" / RAW_XLSX_NAME
    if xlsx.exists():
        print(f"raw workbook: {xlsx.name}")
    else:
        loose = list(root.glob(RAW_XLSX_NAME)) + list(root.glob(f"**/{RAW_XLSX_NAME}"))
        if loose:
            import shutil
            shutil.copy(loose[0], xlsx)
            print(f"copied {loose[0]} -> data-raw/")
        else:
            print(f"NOT FOUND: data-raw/{RAW_XLSX_NAME}\n"
                  "Notebook 1 needs it. Notebooks 2-9 read "
                  "data-processed/cleaned_data.csv instead and are fine "
                  "without it.")

    print(f"\nPROJECT : {os.getcwd()}")
else:
    print("Not in Colab — paths resolve from the project root.")


Mounted at /content/drive
installing: ['catboost']
raw workbook: ghana_dropout_study_M.xlsx

PROJECT : /content/drive/MyDrive/Ghana_Dropout_Project_R02


# Notebook 5b — Data Augmentation: SMOTE vs CTGAN

## What changed from R01

| Change | Reason |
|---|---|
| **CTGAN is fitted on the training fold only, refitted per fold** | R01 generated synthetic data once, outside the fold loop. A generator fitted on rows that later appear in validation is leakage, and a nastier kind than an encoder because a GAN can memorise |
| No test-set scoring | GATE-1(iii). R01 scored the seed-42 partition here too |
| Synthetic rows counted and their provenance logged | Q26 — the augmentation configurations are part of the disclosed combination count |
| Fewer seeds, stated | CTGAN is the only expensive thing in this project. Say so rather than running one seed silently |

**Warning on cost.** CTGAN refitted per fold is genuinely slow — this is the
one place where compute is a real constraint. `N_CTGAN_EPOCHS` and
`CTGAN_SEEDS` are set low deliberately. Raise them and state what you used;
do not quietly fall back to fitting once outside the loop, because that is
the leak.

In [3]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      score_binary, two_level_variance)

banner("NOTEBOOK 5b — SMOTE vs CTGAN")
OUT = run_dir("notebook05b_augmentation")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

CTGAN_SEEDS = SEEDS[:2]        # cost-limited; state this in M8
N_CTGAN_EPOCHS = 100
SYNTH_TARGETS = [250, 500]     # extra minority rows per fold
print(f"seeds {CTGAN_SEEDS}, CTGAN epochs {N_CTGAN_EPOCHS}, "
      f"synthetic targets {SYNTH_TARGETS}")

try:
    from ctgan import CTGAN
    HAVE_CTGAN = True
except ImportError:
    try:
        from sdv.single_table import CTGANSynthesizer as CTGAN
        HAVE_CTGAN = True
    except ImportError:
        HAVE_CTGAN = False
        print("CTGAN unavailable. SMOTE arms will still run; record the "
              "omission in M8 rather than leaving the comparison implied.")

try:
    from imblearn.over_sampling import SMOTE
    HAVE_SMOTE = True
except ImportError:
    HAVE_SMOTE = False

NOTEBOOK 5b — SMOTE vs CTGAN
repo            : /content/drive/MyDrive/Ghana_Dropout_Project_R02
provenance      : NONE — set FREEZE_TAG in config.py before scoring the test set
school_handling : drop
FEATURE SET     : records   (PRIMARY — school records only)
primary metric  : auc_pr
seeds [42, 123], CTGAN epochs 100, synthetic targets [250, 500]


In [5]:
# ---- fold-local augmenters ---------------------------------------------
def smote_fold(X_tr, y_tr, n_extra, seed):
    n_pos = int((y_tr == 1).sum())
    target = n_pos + n_extra
    if target > int((y_tr == 0).sum()):
        target = int((y_tr == 0).sum())
    k = max(1, min(5, n_pos - 1))
    sm = SMOTE(random_state=seed, k_neighbors=k,
               sampling_strategy={1: target, 0: int((y_tr == 0).sum())})
    Xr, yr = sm.fit_resample(X_tr, y_tr)
    return pd.DataFrame(Xr, columns=X_tr.columns), pd.Series(yr)


def ctgan_fold(X_tr, y_tr, n_extra, seed, epochs=N_CTGAN_EPOCHS):
    """Fit CTGAN on the MINORITY ROWS OF THIS TRAINING FOLD ONLY."""
    minority = X_tr[y_tr == 1].reset_index(drop=True)
    if len(minority) < 10:
        return X_tr, y_tr
    np.random.seed(seed)
    model = CTGAN(epochs=epochs, verbose=False)
    model.fit(minority)                           # training-fold rows only
    synth = model.sample(n_extra)
    synth = synth[X_tr.columns]
    Xa = pd.concat([X_tr, synth], ignore_index=True)
    ya = pd.concat([pd.Series(y_tr.to_numpy()),
                    pd.Series(np.ones(len(synth), dtype=int))], ignore_index=True)
    return Xa, ya


CONFIGS = [("original", None, 0)]
if HAVE_SMOTE:
    CONFIGS += [(f"SMOTE+{n}", "smote", n) for n in SYNTH_TARGETS]
if HAVE_CTGAN:
    CONFIGS += [(f"CTGAN+{n}", "ctgan", n) for n in SYNTH_TARGETS]
print("configurations:", [c[0] for c in CONFIGS])
print(f"total model fits: {len(CONFIGS)} x {len(CTGAN_SEEDS)} x "
      f"{N_SPLITS*N_REPEATS} = {len(CONFIGS)*len(CTGAN_SEEDS)*N_SPLITS*N_REPEATS}")
print("Add this to the disclosed combination count in M13 (Q26).")

configurations: ['original', 'SMOTE+250', 'SMOTE+500', 'CTGAN+250', 'CTGAN+500']
total model fits: 5 x 2 x 25 = 250
Add this to the disclosed combination count in M13 (Q26).


In [6]:
# ---- run ----------------------------------------------------------------
from lightgbm import LGBMClassifier
import time

rows, prov = [], []
t0 = time.perf_counter()
for seed in CTGAN_SEEDS:
    for fi, (tr, vl) in enumerate(cv_splits(train_pool, seed), 1):
        X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
            train_pool.iloc[tr], train_pool.iloc[vl])
        for label, kind, n_extra in CONFIGS:
            if kind is None:
                Xa, ya = X_tr, y_tr
            elif kind == "smote":
                Xa, ya = smote_fold(X_tr, y_tr, n_extra, seed)
            else:
                Xa, ya = ctgan_fold(X_tr, y_tr, n_extra, seed)

            model = LGBMClassifier(objective="binary", random_state=seed,
                                   **SHARED_PARAMS)
            model.fit(Xa, ya)
            p = model.predict_proba(X_vl)[:, 1]
            rows.append({"seed": seed, "fold": fi, "arm": label,
                         "augmentation": kind or "none",
                         "n_synthetic": len(ya) - len(y_tr),
                         "n_train_rows": len(ya),
                         **score_binary(y_vl, p)})
            prov.append({"seed": seed, "fold": fi, "config": label,
                         "generator_fitted_on": "training fold minority rows only",
                         "n_synthetic_rows": len(ya) - len(y_tr),
                         "validation_rows_synthetic": 0,
                         "validation_rows": len(y_vl)})
        print(f"    seed {seed} fold {fi}/{N_SPLITS*N_REPEATS} "
              f"[{time.perf_counter()-t0:.0f}s]")

fold_df = pd.DataFrame(rows)
fold_df.to_csv(OUT / "augmentation_fold_scores.csv", index=False)
pd.DataFrame(prov).to_csv(OUT / "synthetic_data_provenance.csv", index=False)
print(f"\n{len(fold_df)} fold-level rows")
print("PROVENANCE: every generator was fitted on training-fold rows only, and "
      "zero synthetic rows entered any validation fold.")

    seed 42 fold 1/25 [44s]
    seed 42 fold 2/25 [63s]
    seed 42 fold 3/25 [82s]
    seed 42 fold 4/25 [100s]
    seed 42 fold 5/25 [119s]
    seed 42 fold 6/25 [138s]
    seed 42 fold 7/25 [157s]
    seed 42 fold 8/25 [175s]
    seed 42 fold 9/25 [193s]
    seed 42 fold 10/25 [213s]
    seed 42 fold 11/25 [231s]
    seed 42 fold 12/25 [249s]
    seed 42 fold 13/25 [267s]
    seed 42 fold 14/25 [285s]
    seed 42 fold 15/25 [305s]
    seed 42 fold 16/25 [324s]
    seed 42 fold 17/25 [342s]
    seed 42 fold 18/25 [362s]
    seed 42 fold 19/25 [382s]
    seed 42 fold 20/25 [401s]
    seed 42 fold 21/25 [420s]
    seed 42 fold 22/25 [439s]
    seed 42 fold 23/25 [459s]
    seed 42 fold 24/25 [477s]
    seed 42 fold 25/25 [499s]
    seed 123 fold 1/25 [522s]
    seed 123 fold 2/25 [541s]
    seed 123 fold 3/25 [559s]
    seed 123 fold 4/25 [579s]
    seed 123 fold 5/25 [598s]
    seed 123 fold 6/25 [620s]
    seed 123 fold 7/25 [641s]
    seed 123 fold 8/25 [658s]
    seed 123 fold 9/25

In [7]:
# ---- results ------------------------------------------------------------
var = two_level_variance(fold_df, PRIMARY_METRIC)
summary = (fold_df.groupby(["arm", "augmentation"])
           .agg(auc_pr=("auc_pr", "mean"), auc_roc=("auc_roc", "mean"),
                recall=("recall", "mean"), precision=("precision", "mean"),
                n_synthetic=("n_synthetic", "mean"))
           .reset_index()
           .merge(var[["arm", "between_seed_sd"]], on="arm")
           .sort_values("auc_pr", ascending=False))
summary.to_csv(OUT / "augmentation_summary.csv", index=False)
print(summary.round(4).to_string(index=False))

base = summary[summary["arm"] == "original"]["auc_pr"].iloc[0]
print(f"\nvs the unaugmented training distribution ({base:.4f}):")
for _, r in summary.iterrows():
    if r["arm"] != "original":
        d = r["auc_pr"] - base
        flag = "" if abs(d) > r["between_seed_sd"] else "  (smaller than its own SD)"
        print(f"  {r['arm']:14s} {d:+.4f}{flag}")

plt.figure(figsize=(8, 4))
o = summary.sort_values("auc_pr")
plt.barh(o["arm"], o["auc_pr"], xerr=o["between_seed_sd"], color="steelblue")
plt.axvline(base, ls="--", c="k", lw=1, label="unaugmented")
plt.xlabel("AUC-PR"); plt.legend(); plt.tight_layout()
plt.savefig(OUT / "figures/augmentation_comparison.png", dpi=200); plt.close()

print("\nR10 reported that neither SMOTE nor CTGAN improved on the raw "
      "distribution. If that survives fold-local generator fitting, it is a "
      "stronger result than before, and it is the correct place to say so.")

write_manifest(OUT, {"notebook": "05b_augmentation", "test_set_scored": False,
                     "seeds": CTGAN_SEEDS, "ctgan_epochs": N_CTGAN_EPOCHS,
                     "configurations": [c[0] for c in CONFIGS],
                     "generator_scope": "training fold only, refitted per fold",
                     "total_fits": int(len(fold_df))})

      arm augmentation  auc_pr  auc_roc  recall  precision  n_synthetic  between_seed_sd
CTGAN+250        ctgan  0.9839   0.9967  0.9233     0.9531        250.0           0.0018
CTGAN+500        ctgan  0.9836   0.9964  0.9146     0.9593        500.0           0.0013
 original         none  0.9831   0.9962  0.9076     0.9680          0.0           0.0014
SMOTE+250        smote  0.9805   0.9941  0.9246     0.9602        250.0           0.0029
SMOTE+500        smote  0.9781   0.9917  0.9346     0.9538        500.0           0.0024

vs the unaugmented training distribution (0.9831):
  CTGAN+250      +0.0008  (smaller than its own SD)
  CTGAN+500      +0.0005  (smaller than its own SD)
  SMOTE+250      -0.0026  (smaller than its own SD)
  SMOTE+500      -0.0050

R10 reported that neither SMOTE nor CTGAN improved on the raw distribution. If that survives fold-local generator fitting, it is a stronger result than before, and it is the correct place to say so.


{'run_dir': 'results/notebook05b_augmentation/20260922T214008Z_records',
 'generated_utc': '2026-09-22T21:56:39Z',
 'git': {'commit': '',
  'branch': '',
  'dirty': False,
  'dirty_paths': [],
  'no_git': True,
  'freeze_tag': ''},
 'host': '21db3cb92c0c',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39',
 'python': '3.13.15',
 'protocol': {'split_seed': 42,
  'test_size': 0.2,
  'seeds': [42, 123, 456, 789, 1024, 2048, 3333, 5555, 7777, 9999],
  'n_splits': 5,
  'n_repeats': 5,
  'primary_metric': 'auc_pr',
  'shared_params': {'n_estimators': 300,
   'num_leaves': 31,
   'learning_rate': 0.05,
   'subsample': 0.8,
   'colsample_bytree': 0.8,
   'verbosity': -1},
  'gamma_reported': 2.0,
  'alpha_reported': 0.75,
  'school_handling': 'drop',
  'feature_set': 'records',
  'active_composites': ['attendance_risk_index'],
  'fairness_threshold': 0.1,
  'drop_derived_duplicates': True,
  'suspect_column_actions': {'social_studies_exam_score': 'keep'}},
 'notebook': '05b_augmentation',
 '

---

## Before you submit

Drive has already saved everything — nothing to push. But two things still
have to happen before submission, and neither is automatic.


In [8]:
# ---- what this run produced, and what is still owed ----
import os, sys
from pathlib import Path

try:
    latest = sorted(Path(OUT).parent.glob("*"))[-1]
    files = sorted(p.relative_to(OUT).as_posix() for p in Path(OUT).rglob("*")
                   if p.is_file())
    print(f"run directory : {Path(OUT).relative_to(REPO)}")
    print(f"files written : {len(files)}")
    for f in files:
        print("   ", f)
except Exception as e:
    print("no run directory recorded in this session:", e)

print("""
────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all of results/

2. SET config.FREEZE_TAG BEFORE SCORING THE TEST SET.
   Without git there is no commit hash to anchor the freeze to. Put a
   fixed dated string in config.py — e.g. "R02-freeze-2026-09-25-1430" —
   at the moment you freeze the configuration, and never revise it.
   Notebook 8 refuses to score the test set until it is set.
────────────────────────────────────────────────────────────────────""")


run directory : results/notebook05b_augmentation/20260922T214008Z_records
files written : 7
    RUN_MANIFEST.json
    augmentation_fold_scores.csv
    augmentation_summary.csv
    environment_versions.csv
    figures/augmentation_comparison.png
    pip_freeze.txt
    synthetic_data_provenance.csv

────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,